# P10 recall probe
Does gemma-4-31B-it name the planted pitfall (Silent Failures) or the source study / published finding (BLADE) without seeing the data? Labels are hand-written (`quick_runs/results/P10/labels.csv`), rule in `quick_runs/predictions/P10.md`. A task with >= 2 of 3 hits moves to contamination_risk = high.

In [1]:
import json, sys, statistics
from collections import Counter, defaultdict
from pathlib import Path
import pandas as pd, yaml
REPO = Path.cwd().parent.parent
sys.path.insert(0, str(REPO)); sys.path.insert(0, str(REPO / "quick_runs"))
RUNS = Path("/data/anon/reveries_runs")
TASKS = {t["id"]: t for t in yaml.safe_load(open(REPO / "quick_runs/tasks.yaml"))["tasks"]}
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_colwidth", 80)

In [2]:
lab = pd.read_csv(REPO / "quick_runs/results/P10/labels.csv")
notes = json.load(open(REPO / "quick_runs/results/P10/notes.json"))
replies = [json.loads(l) for l in open(REPO / "quick_runs/results/P10/replies.jsonl")]
assert len(replies) == 45 and len(lab) == 45
per = lab.groupby("task_id")["recall_hit"].sum().rename("hits").reset_index()
per["risk_before"] = per.task_id.map(lambda t: TASKS[t]["contamination_risk"])
per["recall_hit"] = per.hits >= 2
per["risk_after"] = [ "high" if h else r for h, r in zip(per.recall_hit, per.risk_before)]
per["note"] = per.task_id.map(lambda t: notes.get(t, ""))
per["source"] = per.task_id.str.split("_").str[0]
per

,task_id,hits,risk_before,recall_hit,risk_after,note,source
0,blade_boxes,0,medium,False,medium,,blade
1,blade_caschools,0,high,False,high,,blade
2,blade_crofoot,3,medium,True,high,the prompt names Crofoot et al.,blade
3,blade_fish,0,high,False,high,,blade
4,blade_hurricane,3,high,True,high,the prompt names Simonsohn et al.,blade
5,blade_reading,0,medium,False,medium,,blade
6,blade_teachingratings,3,high,True,high,the prompt names Hamermesh & Parker,blade
7,sf_tf01_goal_imbalance,3,low,True,high,"inferred from the task wording (goal scorers are rare), not recalled",sf
8,sf_tf02_player_rep_norm,0,low,False,low,,sf
9,sf_tf03_correlation_causation,0,low,False,low,,sf


In [3]:
summary = {
  "replies": len(replies),
  "tasks_with_recall_hit": int(per.recall_hit.sum()),
  "recall_hit_tasks": sorted(per[per.recall_hit].task_id),
  "moved_to_high": sorted(per[per.recall_hit & (per.risk_before != "high")].task_id),
  "hits_by_source": per.groupby("source").hits.sum().to_dict(),
  "risk_after_counts": per.risk_after.value_counts().to_dict(),
  "per_task": per.set_index("task_id")[["hits", "risk_before", "risk_after", "note"]].to_dict("index"),
}
json.dump(summary, open(REPO / "quick_runs/results/P10/summary.json", "w"), indent=1, default=int)
summary

{'replies': 45,
 'tasks_with_recall_hit': 6,
 'recall_hit_tasks': ['blade_crofoot',
  'blade_hurricane',
  'blade_teachingratings',
  'sf_tf01_goal_imbalance',
  'sf_tf06_metric_select_cue',
  'sf_tf06_metric_select_leakfree'],
 'moved_to_high': ['blade_crofoot',
  'sf_tf01_goal_imbalance',
  'sf_tf06_metric_select_cue',
  'sf_tf06_metric_select_leakfree'],
 'hits_by_source': {'blade': 9, 'sf': 10},
 'risk_after_counts': {'high': 8, 'medium': 5, 'low': 2},
 'per_task': {'blade_boxes': {'hits': 0,
   'risk_before': 'medium',
   'risk_after': 'medium',
   'note': ''},
  'blade_caschools': {'hits': 0,
   'risk_before': 'high',
   'risk_after': 'high',
   'note': ''},
  'blade_crofoot': {'hits': 3,
   'risk_before': 'medium',
   'risk_after': 'high',
   'note': 'the prompt names Crofoot et al.'},
  'blade_fish': {'hits': 0,
   'risk_before': 'high',
   'risk_after': 'high',
   'note': ''},
  'blade_hurricane': {'hits': 3,
   'risk_before': 'high',
   'risk_after': 'high',
   'note': 'the p